In [20]:
!pip install --upgrade datasets torchvision torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

In [1]:
!pip install transformers datasets accelerate evaluate optuna

In [2]:
import optuna
import numpy as np
import pandas as pd

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [3]:
df=pd.read_csv('/content/ticket_classification.csv')
df=df.dropna()

In [4]:
df=df[['subject','body','queue']]

In [5]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Lowercase the text
    text = text.lower()

    # 2. Remove BOTH actual newlines and literal "\n" text strings
    text = text.replace("\n", " ").replace("\\n", " ").replace("\r", " ")

    # 3. Isolates punctuation marks with spaces
    text = re.sub(r"([.,!?():\"'])", r" \1 ", text)

    # 4. Remove extra spaces and squash down to a single clean space
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
df['text'] = df['subject'].apply(clean_text)+" "+df['body'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 account disruption dear customer support team , i am writing to report a significant problem with the centralized account management portal , which currently appears to be offline . this outage is blocking access to account settings , leading to substantial inconvenience . i have attempted to log in multiple times using different browsers and devices , but the issue persists . could you please provide an update on the outage status a

In [7]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['queue'])
num_classes = len(label_encoder.classes_)

In [8]:
df=df[['text','label']]

In [9]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [10]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [11]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [13]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/19693 [00:00<?, ? examples/s]

Map:   0%|          | 0/4924 [00:00<?, ? examples/s]

In [14]:
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

In [15]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [16]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1
    }

In [17]:
def objective(trial):

    # -------------------------
    # Hyperparameters to tune
    # -------------------------

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        5e-5,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [8, 16, 32]
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        0.0,
        0.1
    )

    num_epochs = trial.suggest_int(
        "num_epochs",
        4,
        9
    )

    # -------------------------
    # Load model
    # -------------------------

    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=df["label"].nunique()
    )

    # -------------------------
    # Training arguments
    # -------------------------

    training_args = TrainingArguments(

        output_dir="./results",

        eval_strategy="epoch",

        save_strategy="no",

        learning_rate=learning_rate,

        per_device_train_batch_size=batch_size,

        per_device_eval_batch_size=batch_size,

        num_train_epochs=num_epochs,

        weight_decay=weight_decay,

        logging_steps=50,

        fp16=True,

        report_to="none"
    )

    # -------------------------
    # Trainer
    # -------------------------

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        compute_metrics=compute_metrics
    )

    # -------------------------
    # Train model
    # -------------------------

    trainer.train()

    # -------------------------
    # Evaluate model
    # -------------------------

    eval_result = trainer.evaluate()

    return eval_result["eval_f1"]

In [18]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=10
)

[I 2026-06-09 12:42:50,723] A new study created in memory with name: no-name-39712cdb-d3b4-4c8d-ba04-2964e496dfdf


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.549795,1.537111,0.436434,0.407426
2,1.349879,1.380738,0.506905,0.483000
3,1.110044,1.271230,0.570877,0.561491
4,0.783668,1.207998,0.626929,0.617874
5,0.622343,1.193200,0.649269,0.642838


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.622343,1.193200,5,0.649269,0.642838


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.526360,1.535815,0.432981,0.400437
2,1.338356,1.382392,0.494517,0.476706
3,0.987201,1.212427,0.595654,0.586420
4,0.684652,1.171492,0.648457,0.642464
5,0.425006,1.187591,0.667344,0.663274


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.425006,1.187591,5,0.667344,0.663274


[I 2026-06-09 14:12:30,594] Trial 9 finished with value: 0.6632741657242992 and parameters: {'learning_rate': 4.199310434372704e-05, 'batch_size': 16, 'weight_decay': 0.011246566219617683, 'num_epochs': 5}. Best is trial 8 with value: 0.7236280247060516.


In [19]:
print(study.best_trial.params)

{'learning_rate': 2.2394835559353657e-05, 'batch_size': 8, 'weight_decay': 0.005931142022441949, 'num_epochs': 9}
